this is the measurement layer of RAG, and honestly, this is where most systems fail silently.

Let’s explain all metrics in the same structured way:

👉 What

👉 Why

👉 Formula (intuitive)

👉 Example

👉 Deep analogy

🧠 1. Why RAG Evaluation Matters

Good answer ≠ Good system

You need to measure:

Layer	          What to Evaluate

Retrieval	      Did we fetch the right docs?

Context	          Is the context useful?

Generation	      Is the answer correct & grounded?

🔵 2. Precision@K

🔍 What is it?

👉 Out of top K retrieved docs, how many are relevant

📊 Formula (intuitive)

Precision@K = Relevant Retrieved / K

🧪 Example

Query:

"What is diabetes?"

Retrieved (K=3):

1. Diabetes affects blood sugar ✔

2. Heart disease causes ❌

3. Insulin resistance ✔

Relevant docs = 2

👉 Precision@3:

2 / 3 = 0.67

🧠 Analogy (Shopping Accuracy)

You buy 3 items:

2 useful

1 useless

👉 Precision = “How many of my picks were actually useful?”

✅ Measures

👉 Quality of retrieved results

🟢 3. Recall@K

🔍 What is it?

👉 Out of all relevant docs, how many did we retrieve

📊 Formula

Recall@K = Retrieved Relevant / Total Relevant

🧪 Example

Relevant docs in system:

A, B, C (3 total)

Retrieved:

A, B

👉 Recall:

2 / 3 = 0.67

🧠 Analogy (Treasure Hunt)

There are 3 treasures buried.

You found 2.

👉 Recall = “How much did you discover?”

✅ Measures

👉 Coverage

🟣 4. NDCG@K (Ranking Quality) ⭐

🔍 What is it?

👉 Measures how well results are ordered

🧠 Key Idea

👉 Higher-ranked relevant docs = better

🧪 Example

Good ranking:

1. Relevant ✔

2. Relevant ✔

3. Irrelevant ❌

Bad ranking:

1. Irrelevant ❌

2. Relevant ✔

3. Relevant ✔

👉 Both have same Precision@3

👉 But first is better ranked

📊 Intuition

Top positions matter more

🧠 Analogy (Google Search)

Would you click:

👉 Relevant result on page 1

OR

👉 Relevant result on page 5

👉 Ranking matters!

✅ Measures

👉 Ranking quality

🔴 5. Faithfulness

🔍 What is it?

👉 Is the answer grounded in retrieved context

❗ Problem it solves

LLMs hallucinate:

Context: "Diabetes affects blood sugar"

Answer: "Diabetes is caused by viruses" ❌

🧪 Example

Context:

"Diabetes affects blood sugar levels"

Answer:

"Diabetes affects blood sugar levels" ✔

👉 Faithfulness = HIGH

🧠 Analogy (Open Book Exam)

Student must answer only using textbook

If they invent new info → wrong

✅ Measures

👉 Hallucination control

🟡 6. Answer Relevancy

🔍 What is it?

👉 Does answer actually answer the question

🧪 Example

Query:

"What causes diabetes?"

Answer:

"Diabetes affects blood sugar levels"

👉 Factually correct BUT ❌ not answering “cause”

🧠 Analogy (Interview Question)

Interviewer asks:

“Why do you want this job?”

You answer:

“I graduated in 2020”

👉 Relevant? ❌

✅ Measures

👉 Question-answer alignment

🟢 7. Context Precision

🔍 What is it?

👉 How much of retrieved context is actually useful

🧪 Example

Context:

1. Diabetes affects blood sugar ✔

2. Stock market trends ❌

3. Insulin resistance ✔

👉 2 useful out of 3

Context Precision = 2/3 = 0.67

🧠 Analogy (Notes Quality)

You prepare notes:

2 important topics

1 irrelevant

👉 Precision = “How clean are my notes?”

✅ Measures

👉 Noise in context

🔵 8. Context Recall

🔍 What is it?

👉 Does context contain all required information

🧪 Example

Golden answer requires:

A + B

Context:

Only A

👉 Missing B → incomplete

🧠 Analogy (Cooking Recipe)

Recipe needs:

salt

spices

You only have salt

👉 Dish incomplete

✅ Measures

👉 Completeness of context

🔥 9. Full Evaluation Picture

User Query

   ↓

Retrieval → Precision@K, Recall@K, NDCG

   ↓

Context → Context Precision, Context Recall

   ↓

Generation → Faithfulness, Answer Relevancy

🧠 10. Deep Insight (Very Important)

Each metric answers a different question:

Metric	                Question

Precision@K	            Are retrieved docs useful?

Recall@K	            Did we miss anything?

NDCG	                Are best docs ranked first?

Context Precision	    Is context clean?

Context Recall	        Is context complete?

Faithfulness	        Is answer grounded?

Answer Relevancy	    Does answer match query?

🔥 11. Combined Analogy (Best Way to Remember)

🎓 Exam System

Step 1: Study Material (Retrieval)

Precision → quality of material

Recall → coverage of syllabus

Step 2: Notes (Context)

Context Precision → no junk notes

Context Recall → all topics covered

Step 3: Exam Answer (Generation)

Faithfulness → based on textbook

Relevancy → answers the question

🎯 Final Summary

Category	             Metrics
Retrieval	             Precision@K, Recall@K, NDCG
Context	                 Context Precision, Context Recall
Generation	             Faithfulness, Answer Relevancy

🚀 Final Insight

👉 Improving RAG is not about one metric:

High Precision + Low Recall → narrow system  

High Recall + Low Precision → noisy system  

Good ranking + Good grounding → great system

In [1]:
import os
import json
import numpy as np
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from sentence_transformers import SentenceTransformer
import chromadb

load_dotenv()
assert os.getenv("OPENAI_API_KEY")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()
collection = client.create_collection("rag_eval")

In [2]:
documents = [
    "Diabetes affects blood sugar levels.",
    "Hypertension increases heart disease risk.",
    "Diversification reduces investment risk.",
    "Neural networks are used in deep learning."
]

collection.add(
    documents=documents,
    embeddings=[embedding_model.encode(d).tolist() for d in documents],
    ids=[str(i) for i in range(len(documents))]
)

# GOLDEN DATASET
golden_data = [
    {
        "query": "What is diabetes?",
        "relevant_docs": ["Diabetes affects blood sugar levels."],
        "answer": "Diabetes affects blood sugar levels."
    },
    {
        "query": "How to reduce investment risk?",
        "relevant_docs": ["Diversification reduces investment risk."],
        "answer": "Diversification reduces investment risk."
    }
]

In [3]:
def retrieve(query, k=2):
    emb = embedding_model.encode(query).tolist()
    results = collection.query(query_embeddings=[emb], n_results=k)
    return results["documents"][0]

Precision@K

In [4]:
def precision_at_k(retrieved, relevant, k):
    retrieved_k = retrieved[:k]
    rel = sum([1 for doc in retrieved_k if doc in relevant])
    return rel / k

Recall@K

In [5]:
def recall_at_k(retrieved, relevant, k):
    retrieved_k = retrieved[:k]
    rel = sum([1 for doc in retrieved_k if doc in relevant])
    return rel / len(relevant)

NDCG@K

In [6]:
def ndcg_at_k(retrieved, relevant, k):
    dcg = 0
    for i, doc in enumerate(retrieved[:k]):
        if doc in relevant:
            dcg += 1 / np.log2(i + 2)

    idcg = sum([1 / np.log2(i + 2) for i in range(min(len(relevant), k))])

    return dcg / idcg if idcg > 0 else 0

In [14]:
import re

def extract_score(text):
    try:
        # Extract first float number
        match = re.search(r"\d*\.?\d+", text)
        if match:
            return float(match.group())
    except:
        pass

    return 0.0  # fallback

Faithfulness

In [18]:
def faithfulness(query, answer, context):
    prompt = f"""
    Score from 0 to 1.

    Return ONLY a number.

    Context:
    {context}

    Answer:
    {answer}
    """

    response = llm.invoke(prompt).content
    return extract_score(response)

Answer Relevancy

In [19]:
def answer_relevancy(query, answer):
    prompt = f"""
    Score from 0 to 1.

    Return ONLY a number.

    Query:
    {query}

    Answer:
    {answer}
    """

    response = llm.invoke(prompt).content
    return extract_score(response)

Context Precision

In [20]:
def context_precision(query, context):
    prompt = f"""
    Score from 0 to 1.

    Return ONLY a number.

    Query:
    {query}

    Context:
    {context}
    """

    response = llm.invoke(prompt).content
    return extract_score(response)

Context Recall

In [21]:
def context_recall(query, context, golden_answer):
    prompt = f"""
    Score from 0 to 1.

    Return ONLY a number.

    Context:
    {context}

    Expected Answer:
    {golden_answer}
    """

    response = llm.invoke(prompt).content
    return extract_score(response)

RAG Generation

In [22]:
def generate_answer(query, context):
    prompt = f"""
    Answer using context only.

    Context:
    {context}

    Query:
    {query}
    """

    return llm.invoke(prompt).content

FULL EVALUATION PIPELINE

In [23]:
def evaluate_rag(golden_data):

    results = []

    for item in golden_data:

        query = item["query"]
        relevant_docs = item["relevant_docs"]
        golden_answer = item["answer"]

        retrieved_docs = retrieve(query, k=2)
        context = "\n".join(retrieved_docs)

        answer = generate_answer(query, context)

        # Retrieval Metrics
        p = precision_at_k(retrieved_docs, relevant_docs, k=2)
        r = recall_at_k(retrieved_docs, relevant_docs, k=2)
        ndcg = ndcg_at_k(retrieved_docs, relevant_docs, k=2)

        # Generation Metrics
        faith = faithfulness(query, answer, context)
        ans_rel = answer_relevancy(query, answer)

        # Context Metrics
        ctx_p = context_precision(query, context)
        ctx_r = context_recall(query, context, golden_answer)

        results.append({
            "query": query,
            "precision@k": p,
            "recall@k": r,
            "ndcg@k": ndcg,
            "faithfulness": faith,
            "answer_relevancy": ans_rel,
            "context_precision": ctx_p,
            "context_recall": ctx_r
        })

    return results

In [24]:
if __name__ == "__main__":
    results = evaluate_rag(golden_data)

    import json
    print(json.dumps(results, indent=2))

[
  {
    "query": "What is diabetes?",
    "precision@k": 0.5,
    "recall@k": 1.0,
    "ndcg@k": 1.0,
    "faithfulness": 0.8,
    "answer_relevancy": 0.8,
    "context_precision": 0.3,
    "context_recall": 0.5
  },
  {
    "query": "How to reduce investment risk?",
    "precision@k": 0.5,
    "recall@k": 1.0,
    "ndcg@k": 1.0,
    "faithfulness": 1.0,
    "answer_relevancy": 0.8,
    "context_precision": 0.8,
    "context_recall": 1.0
  }
]
